# Imports

In [9]:
import os
import glob
import torch
import torchaudio
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
import numpy as np
import librosa

from torchmetrics.classification import BinaryF1Score

In [10]:
# Use GPU if available, else use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [11]:
# Define paths and dataset info (train/val/test split)
SPLIT_ROOT = r"Final_Merged_Dataset\split_70_15_15"
TRAIN_DIR = os.path.join(SPLIT_ROOT, "train")
VAL_DIR = os.path.join(SPLIT_ROOT, "val")
TEST_DIR = os.path.join(SPLIT_ROOT, "test")
CLASSES = ["FAKE", "REAL"]

print(f"Train dir: {TRAIN_DIR}")
print(f"Val dir: {VAL_DIR}")
print(f"Test dir: {TEST_DIR}")
print(f"Classes: {CLASSES}")

Train dir: Final_Merged_Dataset\split_70_15_15\train
Val dir: Final_Merged_Dataset\split_70_15_15\val
Test dir: Final_Merged_Dataset\split_70_15_15\test
Classes: ['FAKE', 'REAL']


# Dataset Class

In [12]:
class AudioDataset(Dataset):
    def __init__(self, data_dir, classes):
        self.file_list = []
        self.labels = []
        for label, class_name in enumerate(classes):
            class_dir = os.path.join(data_dir, class_name)
            if not os.path.exists(class_dir):
                continue
            files = glob.glob(os.path.join(class_dir, "*.mp3"))
            for f in files:
                self.file_list.append(f)
                self.labels.append(label)
                
        # MelSpectrogram transform
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=22050, n_fft=2048, hop_length=512, n_mels=128
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_path = self.file_list[idx]
        
        # Load audio with librosa (avoids TorchCodec dependency)
        waveform_np, sr = librosa.load(file_path, sr=22050, mono=True)
        waveform = torch.from_numpy(waveform_np).unsqueeze(0)
            
        # Pad or truncate to 5 seconds
        target_length = 5 * 22050
        if waveform.shape[1] < target_length:
            waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
        else:
            waveform = waveform[:, :target_length]
            
        # Create Mel Spectrogram
        mel_spec = self.mel_transform(waveform)
        mel_spec_db = self.amplitude_to_db(mel_spec)
        
        # Normalize to [0, 1]
        mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-8)
        
        # Resize for VGG16 (512x512)
        mel_spec_resized = torch.nn.functional.interpolate(
            mel_spec_db.unsqueeze(0), size=(512, 512), mode='bilinear', align_corners=False
        ).squeeze(0)
        
        # Repeat to 3 channels (RGB) for VGG16
        mel_spec_rgb = mel_spec_resized.repeat(3, 1, 1)
        
        return mel_spec_rgb, torch.tensor(self.labels[idx], dtype=torch.float32)

In [13]:
train_dataset = AudioDataset(TRAIN_DIR, CLASSES)
val_dataset = AudioDataset(VAL_DIR, CLASSES)
test_dataset = AudioDataset(TEST_DIR, CLASSES)

batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Loaded train: {len(train_dataset)} items")
print(f"Loaded val: {len(val_dataset)} items")
print(f"Loaded test: {len(test_dataset)} items")

Loaded train: 10164 items
Loaded val: 2177 items
Loaded test: 2181 items


# Architecture of the model

In [14]:
# Clear stale GPU memory from any previous model in the notebook session
if 'model' in globals():
    del model
if 'vgg16' in globals():
    del vgg16
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load VGG16 with ImageNet weights
vgg16 = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

# Unfreeze all base model layers for full fine-tuning
for param in vgg16.features.parameters():
    param.requires_grad = True

# Modify classifier for our binary classes
num_features = vgg16.classifier[6].in_features
vgg16.classifier[6] = nn.Sequential(
    nn.Linear(num_features, 1),
    nn.Sigmoid()
)

# Prefer CUDA, but gracefully fall back to CPU on OOM
try:
    model = vgg16.to(device)
except RuntimeError as e:
    if "out of memory" in str(e).lower() and device.type == "cuda":
        print("CUDA OOM while moving model to GPU. Falling back to CPU.")
        torch.cuda.empty_cache()
        device = torch.device("cpu")
        model = vgg16.to(device)
    else:
        raise

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Device in use: {device}")
print(f"Total trainable parameters: {trainable_params}")
print(f"Total parameters: {total_params}")

Device in use: cuda
Total trainable parameters: 134264641
Total parameters: 134264641


In [15]:
# Training hyperparameters
num_epochs = 50
learning_rate_backbone = 1e-5
learning_rate_head = 1e-4

# Loss function and optimizer
loss_fn = nn.BCELoss()
optimizer = torch.optim.AdamW(
    [
        {"params": model.features.parameters(), "lr": learning_rate_backbone},
        {"params": model.classifier.parameters(), "lr": learning_rate_head},
    ],
    weight_decay=1e-4,
)

def evaluate_model(model, data_loader, loss_fn, device):
    model.eval()
    metric = BinaryF1Score().to(device)
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for inputs_batch, labels_batch in data_loader:
            inputs_batch = inputs_batch.to(device)
            labels_batch = labels_batch.to(device).reshape(-1, 1)

            predictions = model(inputs_batch)
            loss = loss_fn(predictions, labels_batch)

            total_loss += loss.item() * labels_batch.size(0)
            predicted_labels = (predictions >= 0.5).int()
            total_correct += (predicted_labels == labels_batch.int()).sum().item()
            total_samples += labels_batch.size(0)
            metric.update(predictions, labels_batch.int())

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    f1 = metric.compute().item()
    return avg_loss, accuracy, f1

# Training loop
for epoch in range(num_epochs):
    model.train()
    train_metric = BinaryF1Score().to(device)
    epoch_loss = 0.0
    num_batches = 0

    for inputs_batch, labels_batch in train_loader:
        inputs_batch = inputs_batch.to(device)
        labels_batch = labels_batch.to(device).reshape(-1, 1)

        optimizer.zero_grad()
        predictions = model(inputs_batch)
        loss = loss_fn(predictions, labels_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        train_metric.update(predictions, labels_batch.int())
        num_batches += 1

    if num_batches > 0:
        train_loss = epoch_loss / num_batches
        train_f1 = train_metric.compute().item()
        val_loss, val_acc, val_f1 = evaluate_model(model, val_loader, loss_fn, device)

        if epoch % 5 == 0:  # Print every 5 epochs
            print(
                f"Epoch [{epoch+1}/{num_epochs}], "
                f"Train Loss: {train_loss:.4f}, Train F1: {train_f1:.4f}, "
                f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}"
            )
    else:
        print("No training batches processed. Is the train split empty?")

print("Training complete!")

Epoch [1/50], Train Loss: 0.3270, Train F1: 0.8244, Val Loss: 0.2723, Val Acc: 0.9403, Val F1: 0.9083
Epoch [6/50], Train Loss: 0.1171, Train F1: 0.9755, Val Loss: 0.3915, Val Acc: 0.9605, Val F1: 0.9357
Epoch [11/50], Train Loss: 0.1427, Train F1: 0.9802, Val Loss: 0.9807, Val Acc: 0.9637, Val F1: 0.9413
Epoch [16/50], Train Loss: 0.0896, Train F1: 0.9843, Val Loss: 1.1811, Val Acc: 0.9596, Val F1: 0.9361
Epoch [21/50], Train Loss: 0.0953, Train F1: 0.9846, Val Loss: 1.1766, Val Acc: 0.9655, Val F1: 0.9446


KeyboardInterrupt: 

In [16]:
# Final validation and test evaluation on split dataset
def evaluate_model_for_split(model, data_loader, loss_fn, device):
    model.eval()
    metric = BinaryF1Score().to(device)
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for inputs_batch, labels_batch in data_loader:
            inputs_batch = inputs_batch.to(device)
            labels_batch = labels_batch.to(device).reshape(-1, 1)

            predictions = model(inputs_batch)
            loss = loss_fn(predictions, labels_batch)

            total_loss += loss.item() * labels_batch.size(0)
            predicted_labels = (predictions >= 0.5).int()
            total_correct += (predicted_labels == labels_batch.int()).sum().item()
            total_samples += labels_batch.size(0)
            metric.update(predictions, labels_batch.int())

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples
    f1 = metric.compute().item()
    return avg_loss, accuracy, f1

val_loss, val_acc, val_f1 = evaluate_model_for_split(model, val_loader, loss_fn, device)
test_loss, test_acc, test_f1 = evaluate_model_for_split(model, test_loader, loss_fn, device)

print("Validation Metrics")
print(f"Val Loss: {val_loss:.4f}")
print(f"Val Accuracy: {val_acc:.4f}")
print(f"Val F1-Score: {val_f1:.4f}")

print("\nTest Metrics")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1-Score: {test_f1:.4f}")
print(f"Test samples: {len(test_dataset)}")

Validation Metrics
Val Loss: 0.7301
Val Accuracy: 0.9619
Val F1-Score: 0.9375

Test Metrics
Test Loss: 0.7917
Test Accuracy: 0.9665
Test F1-Score: 0.9458
Test samples: 2181
